# QPANN Part 1: Error Generators and Circuit Encoding

This is **Part 1** of the 3-part Quantum Physics-Aware Neural Network (QPANN) tutorial series.

Before we can build a QPANN, we must understand the fundamental representations it uses. In this tutorial, we will explore:
1. **Elementary Error Generators:** How we mathematically represent candidate noise channels.
2. **Enumerating Tracked Generators:** How we choose a subset of "plausible" local errors based on our device's qubit connectivity graph.
3. **Pauli-Correlation (C) and Active (A) Errors:** An advanced look at the other error types in the taxonomy.
4. **Circuit Encoding:** How we convert a `pygsti.circuits.Circuit` into a multi-hot numeric tensor suitable for input to Keras layers.

---
## 1. Pauli and Error Generator Bookkeeping

Under the hood, noise is represented using the elementary-error-generator formalism of the *Taxonomy of Small Errors* (Blume-Kohout et al.). The most common error types are:
* **Hamiltonian (coherent) errors ('H'):** Represented by a single Pauli string (e.g. `H_XI` represents a coherent X-rotation-like error on the first qubit).
* **Stochastic (incoherent) errors ('S'):** Also represented by a single Pauli string (e.g. `S_IX` represents a stochastic Pauli-X flip on the second qubit).

The `errgentools` module (`pygsti.extras.ml.errgentools`) provides essential tools to convert Pauli strings to integers and vice-versa, which acts as our global index bookkeeping. Let's examine some of these functions.

In [1]:
import numpy as np
import random
import pygsti
from pygsti.extras.ml import errgentools as et

# Map a Pauli string to its base-4 integer index
# Direct convention: leftmost character corresponds to qubit 0 (direct string position)
n = 2
ps_idx = et.paulistring_to_index('IX', n)
print(f"Pauli string 'IX' maps to integer index: {ps_idx}")
print(f"Index {ps_idx} maps back to Pauli string: {et.index_to_paulistring(ps_idx, n)!r}")

# Retrieve the global index of a modeled error generator.
# Hamiltonian ('H') ranges from [0, 4**n); Stochastic ('S') ranges from [4**n, 2*4**n).
h_idx = et.error_generator_index('H', ('IX',))
s_idx = et.error_generator_index('S', ('IX',))
print(f"Index of H_IX: {h_idx}")
print(f"Index of S_IX: {s_idx}")

# We can easily round-trip these indices back into error generator descriptors
print(f"Index {s_idx} represents error generator: {et.index_to_error_gen(s_idx, n)}")
print(f"Total possible H+S error generators for {n} qubits: {2 * (4**n)}")

I0000 00:00:1785278927.592424   34802 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785278927.593594   34802 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785278927.678912   34802 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785278929.307796   34802 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

Pauli string 'IX' maps to integer index: 1
Index 1 maps back to Pauli string: 'IX'
Index of H_IX: 1
Index of S_IX: 17
Index 17 represents error generator: ('S', ('IX',))
Total possible H+S error generators for 2 qubits: 32


/workspaces/QPANN/pyGSTi/pygsti/extras/ml/errgentools.py:96: SyntaxWarning: invalid escape sequence '\('
  Number of qubits \(n\).
/workspaces/QPANN/pyGSTi/pygsti/extras/ml/errgentools.py:123: SyntaxWarning: invalid escape sequence '\('
  Number of qubits \(n\).
/workspaces/QPANN/pyGSTi/pygsti/extras/ml/errgentools.py:1125: SyntaxWarning: invalid escape sequence '\('
  Number of qubits \(n\).


---
## 2. Enumerating Local, Low-Weight Error Generators

For a system of $n$ qubits, the total number of possible error generators grows exponentially ($O(4^n)$). To keep a QPANN polynomial in size, we only let the network predict rates for a "plausible" subset of candidate error generators $G$. Typically, this subset consists of:
* All weight-1 error generators (single-qubit coherent and stochastic errors).
* Weight-2 error generators that lie on connected edges of our device's qubit connectivity graph.

To do this, we describe our device's connectivity using a graph Laplacian, and call the convenience entry point `up_to_weight_k_error_gens_from_qubit_graph`. Let's demonstrate this on a 3-qubit line graph (0-1-2) with a hop distance of 1.

In [2]:
from pygsti.processors.processorspec import QubitProcessorSpec as QPS

# 1. Set up a 3-qubit device spec with Gcphase gates available on edges (0,1) and (1,2)
qubit_labels = [0, 1, 2]
availability = {'Gcphase': [(0, 1), (1, 2)]}
pspec = QPS(num_qubits=3, qubit_labels=qubit_labels,
            gate_names=['Gxpi2', 'Gypi2', 'Gcphase'], availability=availability)

# 2. Define the graph Laplacian of the 0-1-2 line connectivity
# Row/column indices map directly to qubit labels
laplacian = np.array([
    [ 1, -1,  0],  # qubit 0 connected to 1
    [-1,  2, -1],  # qubit 1 connected to 0 and 2
    [ 0, -1,  1]   # qubit 2 connected to 1
])

# 3. Enumerate all weight-1 and graph-local weight-2 generators for 'H' and 'S'
modelled_error_generators = et.up_to_weight_k_error_gens_from_qubit_graph(
    k=2, n=3, qubit_graph_laplacian=laplacian, num_hops=1, egtypes=['H', 'S']
)

print(f"Total local error generators enumerated (k=2, hops=1): {len(modelled_error_generators)}")
print("Sample generators:")
for eg in random.sample(modelled_error_generators,10):
    print(" ", eg)

Total local error generators enumerated (k=2, hops=1): 54
Sample generators:
  ('S', ('IXX',))
  ('H', ('IXY',))
  ('S', ('YII',))
  ('H', ('ZXI',))
  ('S', ('IIX',))
  ('S', ('IYZ',))
  ('S', ('IYY',))
  ('S', ('IXI',))
  ('H', ('IYI',))
  ('H', ('IZY',))


---
## 3. Pauli-Correlation (C) and Active (A) Error Generators (Advanced)

While the QPANN paper primarily demonstrates Hamiltonian ('H') and Stochastic ('S') errors, the underlying pyGSTi codebase on this branch is fully unified and supports all four sectors of the error taxonomy:
* **Pauli-Correlation errors ('C'):** Symmetric errors indexed by an unordered pair of two distinct non-identity Paulis (representing correlated stochastic noise).
* **Active errors ('A'):** Antisymmetric errors indexed by an unordered pair of distinct non-identity Paulis (representing coherent/amplitude-damping-like non-unital crosstalk).

Because Active errors are antisymmetric ($A_{P,Q} = -A_{Q,P}$), errgentools provides a sign canonicalization utility that tracks when a swap is needed. Let's see how these are indexed.

In [4]:
# Canonicalize a pair of Paulis lexicographically: 'ZZ' and 'XY' -> 'XY' < 'ZZ' (True means swapped)
P, Q, was_swapped = et.canonical_pauli_pair('ZZ', 'XY')
print(f"Canonical pair: ({P!r}, {Q!r}), swapped={was_swapped}")

# The canonicalization sign is -1 ONLY when we swap the Paulis of an Active ('A') type generator
print("C canonicalization sign (swapped):", et.error_generator_canonicalization_sign('C', ('ZZ', 'XY')))
print("A canonicalization sign (swapped):", et.error_generator_canonicalization_sign('A', ('ZZ', 'XY')))
print("A canonicalization sign (no-swap):", et.error_generator_canonicalization_sign('A', ('XY', 'ZZ')))

# We can easily include 'C' and 'A' types in our device-restricted enumeration
modelled_all_four = et.up_to_weight_k_error_gens_from_qubit_graph(
    k=2, n=3, qubit_graph_laplacian=laplacian, num_hops=1, egtypes=['H', 'S', 'C', 'A']
)
print(f"Total error generators including C/A (k=2, hops=1): {len(modelled_all_four)}")

Canonical pair: ('XY', 'ZZ'), swapped=True
C canonicalization sign (swapped): 1
A canonicalization sign (swapped): -1
A canonicalization sign (no-swap): 1
Total error generators including C/A (k=2, hops=1): 468


---
## 4. Encoding Circuits as Tensors

To feed quantum circuits into our neural network, we must convert each circuit layer into a numeric vector. The standard encoder `StandardCircuitEncoder` maps each layer to a multi-hot binary vector of length `encoder.length`. 

The length of this vector corresponds exactly to the total number of available gates (names + qubit placements) on the processor. A layer containing gates $g_1, g_2, ...$ has $1.0$ at their respective indices, and $0.0$ elsewhere.

Let's build an encoder, encode some circuits, and look at the resulting tensor. We'll use `circuits_to_tensor` to batch multiple encoded circuits into a padded 3D array of shape `(num_circuits, max_depth, encoder.length)`.

In [8]:
from pygsti.extras.ml import encoding
from pygsti.circuits import Circuit

# 1. Define some test circuits
circuits = [
    Circuit('[Gxpi2:0Gypi2:1]@(0,1,2)'),                    # depth 1
    Circuit('[Gcphase:0:1Gypi2:2][Gxpi2:1Gypi2:0]@(0,1,2)')  # depth 2
]

print("Example circuits:")
for i, circuit in enumerate(circuits):
    print(f"  Circuit {i}: ")
    print(f"{circuit}")

# 2. Build the encoder
encoder = encoding.StandardCircuitEncoder(pspec)
print(f"Encoder gate indexing (length {encoder.length}):")
for idx, gate in enumerate(encoder.gate_indexing):
    print(f"  Index {idx} -> {gate}")

# 3. Convert circuits list to a batched tensor
circuits_tensor = encoding.circuits_to_tensor(circuits, encoder)
print(f"\nCircuits tensor shape: {circuits_tensor.shape} (num_circuits, max_depth, encoder.length)")
print("Circuit 0 encoded tensor (depth 1, zero-padded at layer 1):\n", circuits_tensor[0])
print("Circuit 1 encoded tensor (depth 2):\n", circuits_tensor[1])

Example circuits:
  Circuit 0: 
Qubit 0 ---|Gxpi2|---
Qubit 1 ---|Gypi2|---
Qubit 2 ---|     |---

  Circuit 1: 
Qubit 0 ---| C1  |-|Gypi2|---
Qubit 1 ---| C0  |-|Gxpi2|---
Qubit 2 ---|Gypi2|-|     |---

Encoder gate indexing (length 8):
  Index 0 -> Gxpi2:0
  Index 1 -> Gxpi2:1
  Index 2 -> Gxpi2:2
  Index 3 -> Gypi2:0
  Index 4 -> Gypi2:1
  Index 5 -> Gypi2:2
  Index 6 -> Gcphase:0:1
  Index 7 -> Gcphase:1:2

Circuits tensor shape: (2, 2, 8) (num_circuits, max_depth, encoder.length)
Circuit 0 encoded tensor (depth 1, zero-padded at layer 1):
 [[1. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]
Circuit 1 encoded tensor (depth 2):
 [[0. 0. 0. 0. 0. 1. 1. 0.]
 [0. 1. 0. 1. 0. 0. 0. 0.]]


---
### What's Next?
Now that we can represent error generators and encode circuits as tensors, proceed to **[Part 2: Error Propagation and Locality Filters](QPANN-ErrorPropagation.ipynb)** to learn how we propagate these errors through the circuit and filter them using device locality graphs.